# 🤖 Notebook 3 — Modèle avancé : DistilBERT (Transformer)

Dans le notebook précédent, tu as entraîné des modèles de machine learning
classiques avec TF-IDF. Ils fonctionnaient bien, mais avaient une grosse
limitation : ils ne comprennent **ni l'ordre des mots, ni le contexte**.

Par exemple, TF-IDF ne voit aucune différence entre :
- \"Your account **is** suspended\" (ton compte est suspendu)
- \"Your account **is not** suspended\" (ton compte n'est pas suspendu)

Les **Transformers** sont une famille de modèles de deep learning qui
comprennent le contexte. Aujourd'hui, on va utiliser **DistilBERT**,
une version légère et rapide du célèbre modèle BERT.

---

## Ce que tu vas apprendre dans ce notebook

1. Qu'est-ce qu'un **Transformer** et pourquoi c'est puissant.
2. Comment utiliser **DistilBERT** pour classer des emails.
3. Comment **fine-tuner** (adapter) un modèle pré-entraîné.
4. Comment comparer les résultats avec les modèles classiques du Notebook 02.

---

## 1 — C'est quoi un Transformer ?

### L'idée clé : l'attention

Quand tu lis la phrase \"*Click here to verify **your** account*\",
tu comprends que \"your\" se rapporte à \"account\". Un modèle TF-IDF
ne voit que des mots isolés. Un Transformer, lui, utilise un mécanisme
appelé **attention** qui permet à chaque mot de \"regarder\" tous les
autres mots de la phrase pour mieux comprendre le sens.

### BERT et DistilBERT

| | BERT | DistilBERT |
|---|---|---|
| **Créé par** | Google (2018) | Hugging Face (2019) |
| **Taille** | 110 millions de paramètres | 66 millions de paramètres |
| **Vitesse** | Plus lent | ~60% plus rapide |
| **Performance** | Très élevée | ~97% de la performance de BERT |

**DistilBERT** est une version \"distillée\" de BERT : plus petit,
plus rapide, mais presque aussi performant. C'est un excellent choix
pour débuter avec les Transformers.

### Le principe du fine-tuning

DistilBERT a déjà été entraîné sur d'énormes quantités de texte
(Wikipédia, livres, etc.). Il \"comprend\" déjà l'anglais.

On va l'**adapter** (ça s'appelle le *fine-tuning*) à notre tâche
spécifique : détecter le phishing. C'est comme si on prenait quelqu'un
qui parle déjà couramment anglais et on lui apprenait à reconnaître
les arnaques.

---

## 2 — Mise en place

On va utiliser la bibliothèque **Hugging Face Transformers**, qui rend
l'utilisation de ces modèles avancés très simple.

In [ ]:
# Installation des bibliothèques nécessaires (si pas déjà installées)
# Décommente et exécute cette cellule si besoin :

# !pip install transformers datasets accelerate -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "font.size": 13,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

train = pd.read_csv("data/train.csv")
val   = pd.read_csv("data/val.csv")
test  = pd.read_csv("data/test.csv")

print(f"Entra\u00eenement : {len(train):,} emails")
print(f"Validation   : {len(val):,} emails")
print(f"Test         : {len(test):,} emails")

---

## 3 — Tokenisation avec DistilBERT

Les Transformers n'utilisent pas TF-IDF. Ils ont leur propre façon de
transformer le texte en nombres, appelée **tokenisation par sous-mots**.

Au lieu de découper en mots entiers, DistilBERT découpe en **morceaux
de mots** (sub-words). Par exemple :
- \"unhappiness\" → [\"un\", \"##happiness\"]
- \"phishing\" → [\"phi\", \"##shing\"]

Cela permet au modèle de comprendre même des mots qu'il n'a jamais vus.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Voyons comment le tokeniseur découpe un exemple
example = "URGENT: Verify your PayPal account immediately!"
tokens = tokenizer.tokenize(example)
token_ids = tokenizer.encode(example)

print(f"Texte original : {example}")
print(f"Tokens         : {tokens}")
print(f"IDs num\u00e9riques : {token_ids}")
print(f"Nombre de tokens : {len(tokens)}")

### Préparons les données pour DistilBERT

On doit convertir notre DataFrame pandas en un format que Hugging Face
comprend : un `Dataset`.

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train[["text", "label"]])
val_dataset   = Dataset.from_pandas(val[["text", "label"]])
test_dataset  = Dataset.from_pandas(test[["text", "label"]])

print(f"Dataset d'entra\u00eenement : {train_dataset}")
print(f"\nExemple :\n{train_dataset[0]}")

In [ ]:
# On tronque les emails à 256 tokens max pour que ce soit rapide
MAX_LENGTH = 256

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

print("Tokenisation en cours...")
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized   = val_dataset.map(tokenize_function, batched=True)
test_tokenized  = test_dataset.map(tokenize_function, batched=True)
print("\u2713 Tokenisation termin\u00e9e !")

print(f"\nColonnes du dataset tokenis\u00e9 : {train_tokenized.column_names}")
print(f"Forme d'un exemple d'input_ids : {len(train_tokenized[0]['input_ids'])} tokens")

---

## 4 — Chargement du modèle DistilBERT

On charge DistilBERT pré-entraîné et on ajoute une **tête de classification**
dessus : c'est une petite couche supplémentaire qui transforme la
représentation interne du modèle en une décision binaire (sûr / phishing).

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,  # 2 classes : sûr et phishing
)

# Comptons les paramètres
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Mod\u00e8le charg\u00e9 : {MODEL_NAME}")
print(f"Param\u00e8tres totaux     : {n_params:>12,}")
print(f"Param\u00e8tres entra\u00eenables : {n_trainable:>12,}")
print(f"\nC'est {n_params/1e6:.0f} millions de param\u00e8tres !")
print("\u00c0 comparer avec nos mod\u00e8les TF-IDF qui n'avaient que quelques milliers de poids.")

---

## 5 — Entraînement (fine-tuning)

On va entraîner le modèle pendant **2 epochs** (2 passages complets sur
les données d'entraînement). C'est généralement suffisant pour le
fine-tuning car le modèle sait déjà beaucoup de choses.

### Les paramètres importants

| Paramètre | Valeur | Explication |
|---|---|---|
| `num_train_epochs` | 2 | Nombre de fois qu'on parcourt toutes les données |
| `learning_rate` | 2e-5 | Vitesse d'apprentissage (petit = apprentissage prudent) |
| `per_device_train_batch_size` | 16 | Nombre d'emails traités en même temps |
| `weight_decay` | 0.01 | Régularisation pour éviter le sur-apprentissage |

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    """Calcule les m\u00e9triques pendant l'entra\u00eenement."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }

training_args = TrainingArguments(
    output_dir="./distilbert_phishing",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

print("\u2500"*50)
print("  D\u00e9but du fine-tuning de DistilBERT")
print("  Cela peut prendre quelques minutes...")
print("\u2500"*50)

In [ ]:
trainer.train()
print("\n\u2713 Entra\u00eenement termin\u00e9 !")

---

## 6 — Évaluation sur la validation

Regardons comment DistilBERT s'en sort sur l'ensemble de validation.

In [ ]:
val_results = trainer.evaluate(val_tokenized)

print("\n" + "="*50)
print("  R\u00e9sultats de DistilBERT (Validation)")
print("="*50)
print(f"  Accuracy  : {val_results['eval_accuracy']:.4f}  ({val_results['eval_accuracy']*100:.1f}%)")
print(f"  Pr\u00e9cision : {val_results['eval_precision']:.4f}  ({val_results['eval_precision']*100:.1f}%)")
print(f"  Rappel    : {val_results['eval_recall']:.4f}  ({val_results['eval_recall']*100:.1f}%)")
print(f"  F1-score  : {val_results['eval_f1']:.4f}  ({val_results['eval_f1']*100:.1f}%)")

---

## 7 — Évaluation finale sur le test

Comme dans le Notebook 02, on évalue une dernière fois sur l'ensemble de test.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

test_output = trainer.predict(test_tokenized)
y_pred_bert = np.argmax(test_output.predictions, axis=-1)
y_test = test["label"].values

acc  = accuracy_score(y_test, y_pred_bert)
prec = precision_score(y_test, y_pred_bert)
rec  = recall_score(y_test, y_pred_bert)
f1   = f1_score(y_test, y_pred_bert)

print("\n" + "="*50)
print("  R\u00e9sultats de DistilBERT (TEST FINAL)")
print("="*50)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)")
print(f"  Pr\u00e9cision : {prec:.4f}  ({prec*100:.1f}%)")
print(f"  Rappel    : {rec:.4f}  ({rec*100:.1f}%)")
print(f"  F1-score  : {f1:.4f}  ({f1*100:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_bert)
disp = ConfusionMatrixDisplay(cm, display_labels=["S\u00fbr", "Phishing"])
disp.plot(ax=ax, cmap="Purples", colorbar=False)
ax.set_title("Matrice de confusion \u2014 DistilBERT (Test)", fontsize=13)
ax.set_xlabel("Pr\u00e9diction du mod\u00e8le")
ax.set_ylabel("\u00c9tiquette r\u00e9elle")
plt.tight_layout()
plt.show()

print("\nRapport de classification d\u00e9taill\u00e9 :\n")
print(classification_report(y_test, y_pred_bert,
                            target_names=["S\u00fbr", "Phishing"]))

---

## 8 — Comparaison : TF-IDF vs. DistilBERT

Comparons maintenant les résultats de DistilBERT avec le meilleur
modèle TF-IDF du Notebook 02.

> **Note :** pour cette comparaison, entre les résultats du Notebook 02
> manuellement si le kernel a été redémarré.

In [ ]:
# ✏️ REMPLIS ICI les résultats de ton meilleur modèle TF-IDF (Notebook 02)
# Remplace les valeurs ci-dessous par celles que tu as obtenues.
# (Si tu ne les as plus, re-exécute le Notebook 02 et note-les.)

tfidf_results = {
    "nom": "Meilleur TF-IDF (Notebook 02)",
    "accuracy": 0.00,   # à remplacer
    "precision": 0.00,  # à remplacer
    "rappel": 0.00,     # à remplacer
    "f1": 0.00,         # à remplacer
}

bert_results = {
    "nom": "DistilBERT",
    "accuracy": acc,
    "precision": prec,
    "rappel": rec,
    "f1": f1,
}

In [ ]:
all_results = [tfidf_results, bert_results]

comparison = pd.DataFrame({
    "Mod\u00e8le": [r["nom"] for r in all_results],
    "Accuracy": [r["accuracy"] for r in all_results],
    "Pr\u00e9cision": [r["precision"] for r in all_results],
    "Rappel": [r["rappel"] for r in all_results],
    "F1-score": [r["f1"] for r in all_results],
})

display_df = comparison.copy()
for col in ["Accuracy", "Pr\u00e9cision", "Rappel", "F1-score"]:
    display_df[col] = display_df[col].apply(lambda x: f"{x*100:.1f}%")

print("\n" + "\u2500"*60)
print("  COMPARAISON FINALE : TF-IDF vs. DistilBERT (Test)")
print("\u2500"*60 + "\n")
print(display_df.to_string(index=False))

In [ ]:
metrics = ["Accuracy", "Pr\u00e9cision", "Rappel", "F1-score"]
colors = ["#e67e22", "#8e44ad"]

x = np.arange(len(metrics))
width = 0.3

fig, ax = plt.subplots(figsize=(10, 5))

for i, (result, color) in enumerate(zip(all_results, colors)):
    values = [result["accuracy"], result["precision"],
              result["rappel"], result["f1"]]
    bars = ax.bar(x + i*width, [v*100 for v in values], width,
                  label=result["nom"], color=color, edgecolor="white")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val*100:.1f}", ha="center", fontsize=10, fontweight="bold")

ax.set_ylabel("Score (%)")
ax.set_title("TF-IDF vs. DistilBERT \u2014 Ensemble de test", fontsize=14)
ax.set_xticks(x + width/2)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 105)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

---

## 9 — Expérimente : change un paramètre !

Essaie de modifier **un** paramètre et observe l'effet.
Voici quelques idées :

| Ce que tu peux changer | Comment | Ce qui pourrait se passer |
|---|---|---|
| **Nombre d'epochs** | Change `num_train_epochs=2` → `3` | Le modèle apprend plus longtemps (mieux ou sur-apprentissage ?) |
| **Vitesse d'apprentissage** | Change `learning_rate=2e-5` → `5e-5` | Apprentissage plus rapide mais moins stable |
| **Longueur max des tokens** | Change `MAX_LENGTH=256` → `128` | Plus rapide mais perd le contenu des longs emails |

Pour tester, modifie la valeur dans les cellules ci-dessus et
re-exécute à partir de la section 5.

> **Astuce :** change un seul paramètre à la fois pour comprendre
> son effet. Si tu changes tout d'un coup, tu ne sauras pas ce
> qui a fait la différence !

---

## 10 — Testons avec nos propres emails !

Comme dans le Notebook 02, testons le modèle DistilBERT avec des
emails personnalisés.

In [ ]:
import torch

def predict_email(text):
    """Pr\u00e9dit si un email est s\u00fbr ou phishing avec DistilBERT."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       padding=True, max_length=MAX_LENGTH)
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=-1)
    pred = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][pred].item()
    
    return pred, confidence

# ✏️ MODIFIE CES EMAILS ET AJOUTES-EN D'AUTRES !
my_emails = [
    "URGENT: Your account has been compromised! Click here immediately to verify your identity.",
    "Hi team, just a reminder about tomorrow's meeting at 2pm in room B12.",
    "Congratulations! You've won a $500 gift card. Click the link to claim your prize now.",
    "Dear customer, your package has been shipped. Track your delivery at ups.com/track.",
    "Your account is NOT suspended. This is a legitimate notice from your bank.",
]

print("Pr\u00e9dictions de DistilBERT :\n")
for email in my_emails:
    pred, conf = predict_email(email)
    label = "\ud83d\udea8 PHISHING" if pred == 1 else "\u2705 S\u00dbR"
    preview = email[:70] + ("..." if len(email) > 70 else "")
    print(f"  {label}  (confiance: {conf*100:.0f}%)  \u2502  {preview}")

### Question pour toi

- Est-ce que DistilBERT gère mieux le dernier email (\"Your account is NOT suspended\") que le modèle TF-IDF ?
- La confiance du modèle est-elle toujours justifiée ? Peut-on se fier uniquement au pourcentage ?

---

## 11 — Réflexion et conclusion

Écris un court paragraphe répondant à ces questions :

1. **DistilBERT est-il meilleur que TF-IDF ?** De combien ?

2. **Quels sont les avantages de DistilBERT ?**
   (Indice : compréhension du contexte, négation, ordre des mots...)

3. **Quels sont les inconvénients ?**
   (Indice : temps d'entraînement, taille du modèle, besoin d'un GPU...)

4. **Le jeu en vaut-il la chandelle ?** Pour détecter le phishing,
   est-ce que la différence de performance justifie la complexité
   supplémentaire ?

5. **Quelles sont les limites de tous ces modèles ?**
   (Indice : biais du dataset, phishing évolutif, vie privée,
   le modèle devrait-il remplacer ou assister le jugement humain ?)

*Écris ta conclusion ici :*

1. ...
2. ...
3. ...
4. ...
5. ...

---

## Résumé général de la semaine

Félicitations ! Tu as parcouru un chemin impressionnant cette semaine :

| Jour | Ce que tu as fait |
|---|---|
| **Jour 1** | Compris le phishing, ses signaux d'alerte, et la structure du dataset |
| **Jour 2** | Exploré les données : équilibre des classes, longueurs, mots fréquents, motifs |
| **Jour 3** | Entraîné 3 modèles classiques (TF-IDF + Logistic Regression / Naive Bayes / SVM) |
| **Jour 4** | Utilisé un Transformer (DistilBERT) et comparé avec les modèles classiques |
| **Jour 5** | Analyse d'erreurs, comparaisons, limites, et présentation |

### Ce que tu peux retenir

- L'IA peut **aider** à détecter le phishing, mais elle n'est pas parfaite.
- Des modèles **simples** (TF-IDF) peuvent déjà être très efficaces.
- Des modèles **avancés** (Transformers) comprennent mieux le contexte.
- Les **données** sont aussi importantes que le modèle : bruit, biais, équilibre.
- L'IA devrait **assister** le jugement humain, pas le remplacer.
- Les **métriques** (précision, rappel, F1) racontent une histoire
  plus riche que l'accuracy seule.

**Bravo pour cette semaine de travail !** 🎉